# Tutorial: ACA Managed Environment Module Development

**Module**: `Azure/terraform-azurerm-avm-res-app-managedenvironment`  
**Scenario**: Validate and evidence the changes on the fork branch
`kewalaka/fold-tfmodmake-into-module`, which folds `tfmodmake` changes into the module.

This tutorial covers three complementary workflows:

| Workflow | What it does | When to use |
|----------|-------------|-------------|
| **Issue-driven dev** | Agent picks up an upstream GitHub issue, implements the fix, gates via Reviewer, dispatches CI, opens PR | Starting from a GitHub issue |
| **Existing-PR dev** | Agent reads a draft PR's current state, continues from there, dispatches CI, updates PR | Picking up partially-done work on a fork PR |
| **Upgrade test** | Direct `dispatch_upgrade_test` — applies base, plans head, diffs outputs | Validating a fork branch against upstream without the dev loop |

> **ACA scenario**: The `kewalaka/fold-tfmodmake-into-module` branch already has work done
> and an open PR at [kewalaka/terraform-azurerm-avm-res-app-managedenvironment#2](https://github.com/kewalaka/terraform-azurerm-avm-res-app-managedenvironment/pull/2).
> Use **Workflow B (Existing-PR)** below to hand that PR to the agent and continue from the current state.

## 1. Prerequisites

Before running anything:

```bash
# 1. Authenticate GitHub CLI
gh auth login

# 2. Check your AGENT_DISPATCH_TOKEN is set
#    (fine-grained PAT: kewalaka/avm-contributions, Actions RW + Contents R + Metadata R)
echo $AGENT_DISPATCH_TOKEN | cut -c1-10

# 3. Clone your fork of the module
gh repo fork Azure/terraform-azurerm-avm-res-app-managedenvironment \
  --clone --fork-name terraform-azurerm-avm-res-app-managedenvironment
cd terraform-azurerm-avm-res-app-managedenvironment

# 4. Run avm pre-commit to generate the AVM skill file
#    This writes .agents/skills/AVM-Terraform-Development/SKILL.md
#    which the Developer agent loads at runtime.
./avm pre-commit
```

> **Why pre-commit?** The Developer agent reads the module's own AVM skill at runtime
> for module-specific conventions (variable patterns, example structure, etc.).
> Without it, the agent falls back to additive instructions only — it still works,
> but loses module-specific context.

In [ ]:
import os, json, subprocess
from pathlib import Path

# -- Configure these for your environment --
FORK_OWNER  = os.environ.get("FORK_OWNER", "kewalaka")
MODULE_REPO = "Azure/terraform-azurerm-avm-res-app-managedenvironment"
FORK_REPO   = f"{FORK_OWNER}/terraform-azurerm-avm-res-app-managedenvironment"
BASE_REF    = "v0.4.0"                          # upstream release being upgraded from
HEAD_REF    = "kewalaka/fold-tfmodmake-into-module"  # fork branch with changes

# Load .env
from dotenv import load_dotenv
load_dotenv(override=True)

# Verify config
from config import config
print(f"Endpoint:       {(config.project_endpoint or '(not set)')[:50]}")
print(f"Dispatch token: {'set' if os.environ.get('AGENT_DISPATCH_TOKEN') else 'NOT SET — required'}")

r = subprocess.run(["gh", "auth", "status"], capture_output=True, text=True)
print(f"gh auth:        {'ok' if r.returncode == 0 else 'NOT LOGGED IN — run: gh auth login'}")

## 2. Workflow B — Existing-PR Pipeline

Use this when you already have a fork PR with partial work and want the agent to
review the current state, fill gaps, dispatch CI, and iterate.

**ACA scenario command:**
```bash
python main.py dev \
  --upstream-repo Azure/terraform-azurerm-avm-res-app-managedenvironment \
  --pr 1 \
  --fork-owner kewalaka
```

What this does:
1. Fetches PR #1 from `kewalaka/terraform-azurerm-avm-res-app-managedenvironment`
2. Clones the fork at the PR head branch (`kewalaka/fold-tfmodmake-into-module`)
3. Creates an agent-compliant branch (`agent/manual-pr-1-<id>`) from that commit
4. Developer agent reviews current state and continues ("Review existing changes, continue from current state")
5. Reviewer gates the diff, CI dispatched, PR body updated with evidence


In [ ]:
# 2b. Run the existing-PR pipeline (ACA scenario)
import subprocess
from pathlib import Path

FORK_OWNER  = "kewalaka"         # owner of the fork PR
PR_NUMBER   = 1                  # PR number on the fork

cmd = [
    "python3", "../main.py", "dev",
    "--upstream-repo", MODULE_REPO,
    "--pr",            str(PR_NUMBER),
    "--fork-owner",    FORK_OWNER,
]

print("Command:")
print(" ".join(cmd))
print()
print("This will:")
print("  1. Fetch PR metadata from the fork repo")
print("  2. Clone fork at the PR head branch")
print("  3. Create agent branch from that commit")
print("  4. Run Developer agent (continues from current state)")
print("  5. Run Reviewer agent (up to 3 iterations)")
print("  6. Dispatch module-checks CI")
print("  7. Dispatch module-e2e CI")
print("  8. Update PR body with CI evidence")
print()

# Uncomment to actually run:
# result = subprocess.run(cmd, cwd=Path.cwd().parent)
# print(f"Exit code: {result.returncode}")


## 3. Workflow A — Issue-Driven Dev Pipeline

Use this workflow when contributing to upstream AVM: the agent reads the issue,
implements changes on your fork, gates through the Reviewer, dispatches CI to
`avm-contributions`, and opens a draft PR.

### 2a. Find or create the upstream issue

```bash
# List open issues on the upstream module
gh issue list -R Azure/terraform-azurerm-avm-res-app-managedenvironment

# Or create one describing what needs to change
gh issue create \
  -R Azure/terraform-azurerm-avm-res-app-managedenvironment \
  --title "fold tfmodmake changes into module" \
  --body "Description of the changes..."
```

Note the issue number — pass it to the agent below.

In [ ]:
# 2b. Run the dev pipeline
# Replace 99 with your actual issue number
ISSUE_NUMBER = 99

cmd = [
    "python3", "../main.py", "dev",
    "--upstream-repo", MODULE_REPO,
    "--issue",         str(ISSUE_NUMBER),
    "--fork-owner",    FORK_OWNER,
]

print("Command to run:")
print(" ".join(cmd))
print()
print("This will:")
print("  1. Fetch the issue body from GitHub")
print("  2. Ensure your fork exists and is in sync")
print("  3. Clone the fork to ~/.tfdev/ws/<run-id>/")
print("  4. Run Developer agent (reads module AVM skill + additive instructions)")
print("  5. Run Reviewer agent — approves or requests changes (up to 3 iterations)")
print("  6. Dispatch module-checks CI to avm-contributions, wait for result")
print("  7. Dispatch module-e2e CI to avm-contributions, wait for result")
print("  8. Open draft PR on upstream with CI evidence")
print()
print("Estimated time: 5–20 min depending on e2e CI duration")

# Uncomment to actually run:
# result = subprocess.run(cmd, cwd=Path.cwd().parent)
# print(f"Exit code: {result.returncode}")

### 2c. What the pipeline produces

A successful run leaves:

```
~/.tfdev/ws/<run-id>/
  repo/                  # cloned fork with agent branch checked out
  ci/
    checks/summary.json  # pre-commit / linting results
    e2e/summary.json     # apply + idempotency results per example
```

And a draft PR on the upstream repo with managed body sections:

```markdown
<!-- agent:summary -->
What changed and why.
<!-- /agent:summary -->

<!-- agent:evidence -->
CI run links, check conclusions, example results.
<!-- /agent:evidence -->
```

Content outside those markers is yours to edit freely; the agent only regenerates inside them on subsequent pushes.

## 4. Workflow C — Upgrade Test Dispatch

Use this workflow when you already have a fork branch with changes and want
deterministic upgrade evidence: deploy from `base_ref`, plan with `head_ref`,
diff the outputs.

This is the right path for the `kewalaka/fold-tfmodmake-into-module` branch.

In [ ]:
# Check upgrade-tests.yml exists in avm-contributions
r = subprocess.run(
    ["gh", "workflow", "list", "-R", "kewalaka/avm-contributions"],
    capture_output=True, text=True,
)
print(r.stdout)
if "upgrade-tests" not in r.stdout:
    print("WARNING: upgrade-tests.yml not found — check kewalaka/avm-contributions workflows")

In [ ]:
import sys
sys.path.insert(0, str(Path.cwd().parent))

from tools.dispatch_ci import dispatch_upgrade_test, check_dispatch_token

# Verify token first
print(check_dispatch_token())

In [ ]:
# Dispatch the upgrade test
# base_ref: the upstream release consumers are currently using
# head_ref: the fork branch with your changes

print(f"Dispatching upgrade test:")
print(f"  upstream: {MODULE_REPO}")
print(f"  fork:     {FORK_REPO}")
print(f"  from:     {BASE_REF}")
print(f"  to:       {HEAD_REF}")
print()

# This polls until the GHA run completes (can take 30–60 min for e2e)
# result_json = dispatch_upgrade_test(
#     upstream_repo=MODULE_REPO,
#     fork_repo=FORK_REPO,
#     base_ref=BASE_REF,
#     head_ref=HEAD_REF,
# )
# result = json.loads(result_json)
# print(json.dumps(result, indent=2))

print("Uncomment above to run — this will block until CI completes (~30-60 min).")
print("Follow progress at: https://github.com/kewalaka/avm-contributions/actions")

### 3a. Reading the upgrade result

The `dispatch_upgrade_test` return value:

```json
{
  "status": "success",
  "gha_run_id": "12345678",
  "run_url": "https://github.com/kewalaka/avm-contributions/actions/runs/12345678",
  "conclusion": "success",
  "artifacts_dir": "~/.tfdev/ws/<run-id>/ci/upgrade",
  "upgrade_summary": { ... }
}
```

The `artifacts_dir` will contain:

```
upgrade/
  summary.json          # per-example: base_conclusion, head_plan_result, breaking_changes
  upgrade-plan.json     # raw plan diff for UPGRADE.md authoring
  idempotency.json      # post-head-apply plan (should be empty)
```

In [ ]:
# After a completed run, load and inspect the upgrade summary
# Adjust the path to your actual run's artifacts_dir from the result above

ARTIFACTS_DIR = Path.home() / ".tfdev" / "ws" / "REPLACE_WITH_RUN_ID" / "ci" / "upgrade"
summary_path  = ARTIFACTS_DIR / "summary.json"

if summary_path.exists():
    with open(summary_path) as f:
        summary = json.load(f)
    print(json.dumps(summary, indent=2))
else:
    print(f"Summary not found at {summary_path}")
    print("Run the dispatch above first, then update REPLACE_WITH_RUN_ID.")

## 4. Authoring UPGRADE.md

AVM requires an `UPGRADE.md` entry for any breaking change (TFNFR35). Breaking changes
include: variable deletion, type change, default change, output removal, resource rename
without a `moved` block.

The upgrade-plan artifact gives you the raw material; the Dev agent can draft the
entry from it. Minimum structure required by AVM:

```markdown
## Upgrading from v0.4.x to v0.5.x

### Breaking changes

#### `<variable_name>` type changed

**Before:**
```hcl
variable "workload_profiles" { type = list(string) }
```

**After:**
```hcl
variable "workload_profiles" { type = map(object({...})) }
```

**Migration:** ...
```

The Reviewer will **block** the PR if breaking changes are present in the diff
but `UPGRADE.md` has not been updated (check TFNFR35).

## 5. Checklist before opening the upstream PR

```
[ ] module-checks CI green   (linting, pre-commit)
[ ] module-e2e CI green      (apply + idempotency on all examples)
[ ] upgrade-test CI green    (base apply → head plan → post-head-apply idempotency)
[ ] UPGRADE.md updated       (required if any breaking change; Reviewer will catch this)
[ ] PR description includes  (agent:evidence section populated automatically)
[ ] PR opened as draft,      (agent opens draft; flip to ready once all above pass)
```

Once all boxes are checked, flip the PR from draft to ready on GitHub:
```bash
gh pr ready <pr-number> -R Azure/terraform-azurerm-avm-res-app-managedenvironment
```